# 🟢 [SOLUTION] IBM Quantum Safe Beginner Lab: Cryptographic Discovery & CBOM Generation

## 🎯 Learning Objectives
1. Understand the concept of the **Discover** phase in the IBM Quantum Safe Transformation Framework.
2. Build a codebase scanner that automatically parses and identifies cryptographic algorithms in source files.
3. Classify algorithm risk levels against post-quantum standards (RSA vs. ML-KEM, ECDSA vs. ML-DSA).
4. Export cryptographic metadata to a standardized, OWASP CycloneDX-compliant **Cryptographic Bill of Materials (CBOM)** JSON format.

---
## 🧱 Task 1: Complete the Cryptographic Algorithm Risk Classifier
Your first task is to write a classifier function `classify_crypto_algorithm(algorithm_name, key_size)`. This function is critical for evaluating whether an algorithm is quantum-vulnerable, categorizing its NIST post-quantum category, and recommending the appropriate upgrade path.

In [ ]:
from typing import Dict, Any, List
import json

def classify_crypto_algorithm(algorithm_name: str, key_size: int = 0) -> Dict[str, Any]:
    """
    Given an algorithm name (e.g. 'RSA', 'AES', 'ML-KEM-768', '3DES', 'ECDSA'):
    Return a dictionary with:
    - 'is_quantum_vulnerable': bool
    - 'nist_standard_category': str ('FIPS 203', 'FIPS 204', 'Legacy', 'Deprecated', or 'Quantum Safe')
    - 'recommended_action': str
    """
    algo = algorithm_name.upper().strip()
    
    if "RSA" in algo or "DIFFIE" in algo or "DH" in algo:
        return {
            "is_quantum_vulnerable": True,
            "nist_standard_category": "Legacy",
            "recommended_action": "Migrate to ML-KEM"
        }
    elif "ECDSA" in algo or "ECC" in algo or "SECP" in algo:
        return {
            "is_quantum_vulnerable": True,
            "nist_standard_category": "Legacy",
            "recommended_action": "Migrate to ML-DSA"
        }
    elif "3DES" in algo or "DES" in algo:
        return {
            "is_quantum_vulnerable": True,
            "nist_standard_category": "Deprecated",
            "recommended_action": "Upgrade to AES-256"
        }
    elif "AES" in algo:
        if key_size < 256 and key_size > 0:
            return {
                "is_quantum_vulnerable": True,
                "nist_standard_category": "Legacy",
                "recommended_action": "Upgrade to AES-256"
            }
        else:
            return {
                "is_quantum_vulnerable": False,
                "nist_standard_category": "Quantum Safe",
                "recommended_action": "None"
            }
    elif "ML-KEM" in algo or "KYBER" in algo:
        return {
            "is_quantum_vulnerable": False,
            "nist_standard_category": "FIPS 203",
            "recommended_action": "None"
        }
    elif "ML-DSA" in algo or "DILITHIUM" in algo:
        return {
            "is_quantum_vulnerable": False,
            "nist_standard_category": "FIPS 204",
            "recommended_action": "None"
        }
    else:
        return {
            "is_quantum_vulnerable": True,
            "nist_standard_category": "Legacy",
            "recommended_action": "Migrate to ML-KEM"
        }

---
## 🧱 Task 2: Build the Codebase Discovery Scanner & CBOM Generator
Your second task is to write a scanner that searches through target source code lines, extracts algorithms, runs them through your classifier, and compiles a compliant CycloneDX CBOM JSON structure.

In [ ]:
def scan_codebase_and_generate_cbom(source_lines: List[str]) -> Dict[str, Any]:
    """
    Scans a list of source code strings for cryptographic keywords,
    classifies them, and returns a structured CycloneDX-compliant CBOM dictionary.
    """
    keywords = ["RSA", "ECDSA", "3DES", "AES", "ML-KEM", "ML-DSA"]
    detected_assets = []
    
    for line_idx, line in enumerate(source_lines):
        for kw in keywords:
            if kw in line.upper():
                key_size = 0
                for sz in [4096, 2048, 1024, 256, 192, 128]:
                    if str(sz) in line:
                        key_size = sz
                        break
                
                classification = classify_crypto_algorithm(kw, key_size)
                
                asset_item = {
                    "algorithm": kw,
                    "key_size": key_size,
                    "line_number": line_idx + 1,
                    "is_quantum_vulnerable": classification["is_quantum_vulnerable"],
                    "nist_standard_category": classification["nist_standard_category"],
                    "recommended_action": classification["recommended_action"]
                }
                detected_assets.append(asset_item)
                
    cbom_json = {
        "bomFormat": "CycloneDX",
        "specVersion": "1.4",
        "metadata": {
            "component": {
                "type": "application",
                "name": "Standard Legacy Cryptographic System"
            }
        },
        "components": detected_assets
    }
    return cbom_json

---
## 🧪 Task 3: Test Your Implementation
Run the cell below to verify your implementation on a sample legacy codebase.

In [ ]:
sample_source_code = [
    "import hashlib",
    "# We are using legacy encryption for communication",
    "session_key = generate_key('RSA', 2048)",
    "signature = sign_payload('ECDSA', payload_bytes)",
    "# Future proofing with post-quantum ML-KEM-768",
    "pq_key = generate_key('ML-KEM', 768)",
    "# Weak symmetric cipher to upgrade",
    "legacy_cipher = encrypt_data('3DES', raw_text)",
    "secure_cipher = encrypt_data('AES', 256)"
]

cbom = scan_codebase_and_generate_cbom(sample_source_code)
print(json.dumps(cbom, indent=2))